# AI Research Agent — Evaluation Notebook

**Purpose:**  
This notebook evaluates the performance of the AI Research Agent system under different research depth configurations.

We compare:

- **Standard Mode** — 2 discovery iterations  
- **Deep Mode** — 3 discovery iterations (additional coverage refinement)

The goal is to measure whether deeper iterative research improves report quality.

This notebook focuses on **experimental comparison**, not internal pipeline tracing.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\Bluepal\Desktop\AI_RESEARCH_AGENT


In [2]:
from src.controller.run import run_pipeline
from src.vector_store.client import VectorStoreClient
from src.trace.research_trace import ResearchTrace

c:\Users\Bluepal\Desktop\AI_RESEARCH_AGENT\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Bluepal\Desktop\AI_RESEARCH_AGENT\venv\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
c:\Users\Bluepal\Desktop\AI_RESEARCH_AGENT\src\analytics\agreement_detector.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as 

In [3]:
EVAL_QUERY = "Impact of generative AI on software engineering roles and workforce dynamics"
EVAL_QUERY

'Impact of generative AI on software engineering roles and workforce dynamics'

In [4]:
vector_client = VectorStoreClient(embedding_dim=384)

trace_std = ResearchTrace()

_, _, report_std, _, eval_std = run_pipeline(
    user_query=EVAL_QUERY,
    mode="standard",
    vector_client=vector_client,
    trace=trace_std,
)

eval_std

{'overall_score': 7.5,
 'accuracy': {'score': 9,
  'notes': 'Most claims are supported by the provided summaries, but some sentences are repetitive and lack specific citations.'},
 'completeness': {'score': 6,
  'notes': 'The report addresses some planned research dimensions, but dimensions such as Skill Requirements Shift, Ethical and Societal Implications, and Economic and Market Impact are missing or weakly covered.'},
 'citation_quality': {'score': 8,
  'notes': 'Declarative sentences are properly cited, but some citations are missing or lack specific page numbers.'},
 'structure': {'score': 7,
  'notes': 'The report has a logical flow, but some sections are repetitive and lack depth.'},
 'limitations': ['Lack of coverage of some planned research dimensions',
  'Repetitive sentences and lack of specific citations'],
 'confidence_level': 'medium'}

In [5]:
trace_deep = ResearchTrace()

_, _, report_deep, _, eval_deep = run_pipeline(
    user_query=EVAL_QUERY,
    mode="deep",
    vector_client=vector_client,
    trace=trace_deep,
)

eval_deep

{'overall_score': 8.5,
 'accuracy': {'score': 9,
  'notes': 'All claims are supported by the provided summaries.'},
 'completeness': {'score': 8,
  'notes': 'Most planned research dimensions are addressed, but some are weakly covered or unevenly developed.'},
 'citation_quality': {'score': 9,
  'notes': 'Declarative sentences are properly cited, and citation markers correspond correctly to the references.'},
 'structure': {'score': 8,
  'notes': 'Logical flow and coherence are generally maintained, but some sections could be more balanced.'},
 'limitations': ['Some planned research dimensions are weakly covered or unevenly developed.',
  'The report could benefit from more in-depth analysis and discussion of the findings.'],
 'confidence_level': 'high'}

In [6]:
df = pd.DataFrame([
    {
        "Mode": "Standard",
        "Overall Score": eval_std["overall_score"],
        "Accuracy": eval_std["accuracy"]["score"],
        "Completeness": eval_std["completeness"]["score"],
        "Citation Quality": eval_std["citation_quality"]["score"],
        "Structure": eval_std["structure"]["score"],
    },
    {
        "Mode": "Deep",
        "Overall Score": eval_deep["overall_score"],
        "Accuracy": eval_deep["accuracy"]["score"],
        "Completeness": eval_deep["completeness"]["score"],
        "Citation Quality": eval_deep["citation_quality"]["score"],
        "Structure": eval_deep["structure"]["score"],
    },
])

df

,Mode,Overall Score,Accuracy,Completeness,Citation Quality,Structure
0,Standard,7.5,9,6,8,7
1,Deep,8.5,9,8,9,8


## Experimental Observations

The comparison shows how research depth affects report quality.

**Key trends observed:**

- Deep mode typically improves **completeness**, as additional coverage refinement explores more research dimensions.
- Accuracy and citation quality remain high in both modes due to strict grounding in collected summaries.
- Structural quality is stable across modes, as report synthesis uses a consistent template.
- Overall score often increases in Deep mode, indicating stronger coverage and synthesis.

**Trade-offs:**

| Aspect | Standard Mode | Deep Mode |
|--------|---------------|-----------|
| Iterations | 2 | 3 |
| Runtime | Lower | Higher |
| API Cost | Lower | Higher |
| Coverage | Moderate | Broader |

This experiment demonstrates that iterative refinement improves research breadth and synthesis depth.

## Evaluation Limitations

- Evaluation is performed by an LLM evaluator, which introduces probabilistic judgment.
- Results may vary slightly across runs due to nondeterministic LLM behavior.
- Web search results can change over time.
- Only one research query is tested; broader benchmarking could further validate findings.